# 07 - Data Wrangling and Cleaning
Real data is messy: missing values, duplicates, wrong types, spelling differences, impossible values.
Cleaning it is **step 1** of every EDA.

In [1]:
import pandas as pd

## Step 0: Look at the messy data

In [2]:
df = pd.read_csv("data/messy_data.csv")
df

,name,dept,gender,age,marks,attendance,city,join_date
0,arun,CSE,M,21.0,78.0,92.0,chennai,2022-06-15
1,Bala,ece,M,22.0,65.0,80.0,MADURAI,2022-06-16
2,Chitra,CSE,F,21.0,NaN,95.0,Chennai,2022-06-15
3,Divya,IT,F,NaN,72.0,85.0,Coimbatore,2022-06-17
4,Elango,MECH,M,23.0,55.0,70.0,trichy,2022-06-18
5,Elango,MECH,M,23.0,55.0,70.0,trichy,2022-06-18
6,Fathima,ECE,F,21.0,91.0,98.0,Chennai,2022-06-15
7,Gokul,IT,M,22.0,60.0,NaN,Madurai,2022-06-19
8,Hema,cse,F,21.0,84.0,90.0,Coimbatore,2022-06-16
9,Imran,MECH,M,22.0,480.0,65.0,Trichy,2022-06-20


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 15 entries, 0 to 14
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   name        15 non-null     str    
 1   dept        15 non-null     str    
 2   gender      15 non-null     str    
 3   age         14 non-null     float64
 4   marks       14 non-null     float64
 5   attendance  14 non-null     float64
 6   city        14 non-null     str    
 7   join_date   15 non-null     str    
dtypes: float64(3), str(5)
memory usage: 1.1 KB


## 1. Find missing values

In [4]:
df.isnull().sum()       # count of missing values in each column

name          0
dept          0
gender        0
age           1
marks         1
attendance    1
city          1
join_date     0
dtype: int64

## 2. Handle missing values
Two options: **fill** them or **drop** the rows.

In [5]:
# small example first
small = pd.DataFrame({"name": ["A", "B", "C", "D"], "marks": [80, None, 60, None]})
small

,name,marks
0,A,80.0
1,B,NaN
2,C,60.0
3,D,NaN


In [6]:
small["marks"].fillna(0)

0    80.0
1     0.0
2    60.0
3     0.0
Name: marks, dtype: float64

In [7]:
small["marks"].fillna(small["marks"].mean())     # fill with average

0    80.0
1    70.0
2    60.0
3    70.0
Name: marks, dtype: float64

In [8]:
small.dropna()

,name,marks
0,A,80.0
2,C,60.0


Apply to our data:

In [9]:
df["marks"] = df["marks"].fillna(df["marks"].median())
df["age"] = df["age"].fillna(df["age"].median())
df["attendance"] = df["attendance"].fillna(df["attendance"].mean())
df["city"] = df["city"].fillna("Unknown")
df.isnull().sum()

name          0
dept          0
gender        0
age           0
marks         0
attendance    0
city          0
join_date     0
dtype: int64

## 3. Remove duplicate rows

In [10]:
print("Duplicates:", df.duplicated().sum())
df = df.drop_duplicates()
print("Rows now  :", len(df))

Duplicates: 1
Rows now  : 14


## 4. Clean text (spaces and upper/lower case)

In [11]:
df["name"] = df["name"].str.strip().str.title()
df["dept"] = df["dept"].str.upper()
df["city"] = df["city"].str.title()
df[["name", "dept", "city"]]

,name,dept,city
0,Arun,CSE,Chennai
1,Bala,ECE,Madurai
2,Chitra,CSE,Chennai
3,Divya,IT,Coimbatore
4,Elango,MECH,Trichy
6,Fathima,ECE,Chennai
7,Gokul,IT,Madurai
8,Hema,CSE,Coimbatore
9,Imran,MECH,Trichy
10,Janani,IT,Unknown


## 5. Fix wrong values (outliers / typos)

In [12]:
df.describe()

,age,marks,attendance
count,14.000000,14.000000,14.000000
mean,36.500000,102.714286,84.693878
std,55.696637,109.233292,10.293168
min,21.000000,55.000000,65.000000
25%,21.000000,66.000000,80.500000
50%,22.000000,75.000000,85.500000
75%,22.000000,83.250000,91.500000
max,230.000000,480.000000,99.000000


Marks of 480 and age of 230 are clearly typing mistakes.

In [13]:
df.loc[df["marks"] == 480, "marks"] = 48
df.loc[df["age"] == 230, "age"] = 23
df[["name", "age", "marks"]].describe()

,age,marks
count,14.000000,14.000000
mean,21.714286,71.857143
std,0.726273,13.682426
min,21.000000,48.000000
25%,21.000000,61.250000
50%,22.000000,73.000000
75%,22.000000,80.250000
max,23.000000,95.000000


## 6. Fix data types

In [14]:
df["age"] = df["age"].astype(int)
df["marks"] = df["marks"].astype(int)
df["join_date"] = pd.to_datetime(df["join_date"])
df.dtypes

name                     str
dept                     str
gender                   str
age                    int64
marks                  int64
attendance           float64
city                     str
join_date     datetime64[us]
dtype: object

In [15]:
# dates are now easy to work with
df["join_month"] = df["join_date"].dt.month_name()
df[["name", "join_date", "join_month"]].head()

,name,join_date,join_month
0,Arun,2022-06-15,June
1,Bala,2022-06-16,June
2,Chitra,2022-06-15,June
3,Divya,2022-06-17,June
4,Elango,2022-06-18,June


## 7. Merge two tables (like a SQL join)

In [16]:
hostel = pd.DataFrame({
    "name": ["Arun", "Chitra", "Divya", "Hema"],
    "hostel": ["A-Block", "B-Block", "B-Block", "A-Block"],
})
merged = pd.merge(df, hostel, on="name", how="left")   # keep all students
merged[["name", "dept", "hostel"]].head(8)

,name,dept,hostel
0,Arun,CSE,A-Block
1,Bala,ECE,NaN
2,Chitra,CSE,B-Block
3,Divya,IT,B-Block
4,Elango,MECH,NaN
5,Fathima,ECE,NaN
6,Gokul,IT,NaN
7,Hema,CSE,A-Block


Students without a hostel show `NaN` - fill them:

In [17]:
merged["hostel"] = merged["hostel"].fillna("Day scholar")
merged[["name", "hostel"]].head(8)

,name,hostel
0,Arun,A-Block
1,Bala,Day scholar
2,Chitra,B-Block
3,Divya,B-Block
4,Elango,Day scholar
5,Fathima,Day scholar
6,Gokul,Day scholar
7,Hema,A-Block


## 8. Reset the index and save the clean file

In [18]:
df = df.reset_index(drop=True)
df.to_csv("data/clean_data.csv", index=False)
print("Saved!", df.shape)

Saved! (14, 9)


## Try it yourself
1. Re-load `messy_data.csv` and clean it again, but this time use `dropna()` instead of `fillna()`. How many rows are left?
2. Count the students per cleaned `dept`.
3. Which student joined first? (hint: `sort_values("join_date")`)

In [19]:
# your code here